# 01.3 自动微分（Autograd）

`Autograd` 是 `PyTorch` 的核心能力之一。  

如果你把这一节真正弄明白，后面的 `loss.backward()` 就不再只是“照着写”。  

重点概念

- 计算图（computational graph）
- 梯度（gradients）
- 梯度累积（gradient accumulation）
- `no_grad()` 与 `detach()`

## 学习目标

学完后你应该能

1. 理解什么是 `requires_grad=True`
2. 看懂简单计算图
3. 用 `backward()` 求梯度
4. 理解梯度为什么会累积
5. 正确使用 `torch.no_grad()`
6. 理解 `detach()` 的作用

In [ ]:
import torch

## 1. `requires_grad` 与计算图

当一个张量设置了 `requires_grad=True`，`PyTorch` 会开始追踪它参与的运算。  

这些运算连接起来，就形成了计算图  


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1

print("x =", x)
print("y =", y)
print("x.requires_grad =", x.requires_grad)
print("y.requires_grad =", y.requires_grad)
print("y.grad_fn =", y.grad_fn)

这里可以先从直觉理解：  

- `x` 是需要求导的输入
- `y` 是由 `x` 经过一系列运算得到的结果
- `grad_fn` 表示 `y` 背后有可追踪的计算历史

## 2. `backward()` 求梯度

scalar，直接调用 `backward()` 即可。  


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1
y.backward()

print("x.grad =", x.grad)

手算验证

- `y = x^2 + 3x + 1`
- 当 `x=2` 时，梯度是 `7`

所以 `x.grad == 7`。  


In [ ]:
# 练习 1
# 令 y = 4x^2 - x
# Let y = 4x^2 - x
#
# 1. 令 x=3, requires_grad=True
# 2. 用 backward() 求梯度
# 3. 打印 x.grad

# x =
# y =
# y.backward()
# print(x.grad)

In [ ]:
# 练习 1 参考答案

x = torch.tensor(3.0, requires_grad=True)
y = 4 * x ** 2 - x
y.backward()
print(x.grad)

# 手算

## 3. 非标量输出

如果输出不是标量，`backward()` 需要额外提供梯度权重  


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2

y.backward(gradient=torch.ones_like(y))
print("x.grad =", x.grad)

这里传入 `torch.ones_like(y)`，相当于把每个输出项都等权相加。  

在实际训练里，`loss` 通常已经是标量，所以更常见的是直接：  

- `loss.backward()`

## 4. 梯度累积

这是一个非常关键、非常常见、也非常容易踩坑的点。  

在 `PyTorch` 中，梯度默认会累积到 `.grad` 里。  


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x ** 2
y1.backward()
print("第一次 backward 后 / after first backward:", x.grad)

y2 = 3 * x
y2.backward()
print("第二次 backward 后 / after second backward:", x.grad)

为什么会这样

- 第一次：`d(x^2)/dx = 2x = 4`
- 第二次：`d(3x)/dx = 3`
- 累积结果：`4 + 3 = 7`

这就是训练循环里经常要写 `optimizer.zero_grad()` 的原因之一。  


In [ ]:
# 练习 2
# 复现实验：
# Reproduce the experiment:
# 1. x=1, requires_grad=True
# 2. 先对 y=x^3 backward
# 3. 再对 z=2x backward
# 4. 观察 x.grad 的累积结果

# x =
# y =
# y.backward()
# z =
# z.backward()
# print(x.grad)

In [ ]:
# 练习 2 参考答案

x = torch.tensor(1.0, requires_grad=True)
y = x ** 3
y.backward()
z = 2 * x
z.backward()
print(x.grad)

# dy/dx = 3, dz/dx = 2, total = 5

## 5. 手动清零梯度

在优化参数前，通常要先把旧梯度清零。  


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()
print("清零前 / before zeroing:", x.grad)

x.grad.zero_()
print("清零后 / after zeroing:", x.grad)

z = 3 * x
z.backward()
print("再次 backward 后 / after backward again:", x.grad)

## 6. `torch.no_grad()` / `torch.no_grad()`

有些场景你不想让 `PyTorch` 跟踪梯度，比如：  

- 推理（inference）
- 验证（validation）
- 单纯做数值查看（plain numeric inspection）

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = x * 5

print("y =", y)
print("y.requires_grad =", y.requires_grad)

`no_grad()` 的主要价值

- 节省显存和计算图开销
- 避免不必要的梯度追踪（avoids unnecessary gradient tracking）

## 7. `detach()` / `detach()`

`detach()` 会返回一个新的张量视图，它共享数据，但不再参与当前计算图。  


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = 2 * x
z = y.detach()

print("y.requires_grad =", y.requires_grad)
print("z.requires_grad =", z.requires_grad)

y_sum = y.sum()
y_sum.backward()
print("x.grad =", x.grad)

直觉上可以理解为：  

- `y` 还连接在图里
- `z` 从图里“剪下来”了

In [ ]:
# 练习 3
# 判断下面哪些张量会追踪梯度。
# Decide which tensors below will track gradients.
#
# x = torch.tensor(2.0, requires_grad=True)
# a = x * 2
# with torch.no_grad():
#     b = x * 3
# c = a.detach()
#
# print(a.requires_grad)
# print(b.requires_grad)
# print(c.requires_grad)

In [ ]:
# 练习 3 参考答案

x = torch.tensor(2.0, requires_grad=True)
a = x * 2
with torch.no_grad():
    b = x * 3
c = a.detach()

print(a.requires_grad)
print(b.requires_grad)
print(c.requires_grad)

## 8. 一个最小训练感知例子

下面这个例子还不是真正的训练循环，但它已经很接近了。  


In [ ]:
w = torch.tensor(0.5, requires_grad=True)
x = torch.tensor(2.0)
target = torch.tensor(4.0)

pred = w * x
loss = (pred - target) ** 2
loss.backward()

print("pred =", pred.item())
print("loss =", loss.item())
print("w.grad =", w.grad.item())

这里已经出现了训练最核心的链条：  

- 参数
- 前向计算
- 损失
- 反向传播

下一节学训练循环时，这条链会变成完整版本。  


## 9. 小结

你现在应该能回答

1. `requires_grad=True` 到底意味着什么？
2. 为什么标量输出可以直接 `backward()`？
3. 为什么 `.grad` 会累积？
4. `no_grad()` 和 `detach()` 的区别是什么？
5. 为什么训练中常常先清零梯度？

下一步建议

- `DataLoader`，或者直接进入 `nn.Module` 和训练循环